# Module 2: Epidemic Modeling Template

## Team Members:
*(Fill in)*

## Project Title:
*(Fill in)*

## Project Goal:
This project seeks to... *(what is the purpose of your project -- i.e., describe the question that you seek to answer by analyzing data.)*

## 1. Data and disease background
You can fill out this section throughout the module as you uncover more information about the mystery disease.

By the end of the module (when submitting), you should have some information about each of the following points:
* Prevalence & incidence in the UVA population
* Economic burden (you can generalize from respiratory viruses)
* Symptoms
* Biological mechanisms (anatomy, organ physiology, cell & molecular physiology - you can generalize from viral biology)


## 2. Data Analysis
This section should be filled out sequentially as a full report of the work you've done over this module. You can copy and paste code from any main.py file here, and run it to produce plots. Once you gain more information throughout the module, you do not need to go back and "fix" earlier results. In other words, if your early predictions are found to be wrong when gaining new data, do not go back and rewrite them.

### 2a. Methods

*IN A SUMMARY, DESCRIBE THE METHODS YOU USED TO ANALYZE AND MODEL THE DATA.*


#β = 0.27
#γ = 0.14
#σ = 0.20


<div style="
    border-left: 6px solid #fbc02d;
    background-color: #fff8e1;
    padding: 10px 15px;
    border-radius: 4px;
">
<b style="color:#f57f17;">ANALYSIS AFTER DATA RELEASE #1</b> 

</div>



### 2b. Plot the data & estimate initial growth rate (R0) from early data (through day 45)
This section should come from your python code after Data Release #1.



In [ ]:
# exploratory_analysis_day1.py
import pandas as pd

import matplotlib.pyplot as plt

# Load the data
data = pd.read_csv('Data/mystery_virus_daily_active_counts_RELEASE#1.csv', parse_dates=['date'], header=0, index_col=None)

# Make a plot of the active cases over time

active_col = data.columns[2]
data[active_col] = pd.to_numeric(data[active_col], errors="coerce")

# Day and active infections
days = data["day"]
active_cases = data[active_col]

# Plot the data
plt.figure()
plt.plot(days, active_cases)
plt.xlabel("Day")
plt.ylabel("Active Infections")
plt.title("Day vs Active Infections (Data Release #1)")
plt.show()

# What do you notice about the initial infections? For the first several days, the amount of infections stayed low, curve a little flat. After that, the curve begins to rise faster - exponential-like growth
# How could we measure how quickly its spreading? (cases today - cases yesterday)/casees yesterday - look at the slope of the curve and how long the cases takes to double 
# What information about the virus would be helpful in determining the shape of the outbreak curve? how easily it spreads, how long it takes before the symptoms appear, how long someone can spread it, reecovery time, immmunity rates, and what is being done to prevent it from further spreading


# exploratory_analysis_day2.py

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit

# Load the data
data = pd.read_csv('Data/mystery_virus_daily_active_counts_RELEASE#2.csv', parse_dates=['date'], header=0, index_col=None)

t_data = data['day'].values
y_data = data['active reported daily cases'].values
# We have day number, date, and active cases. We can use the day number and active cases to fit an exponential growth curve to estimate R0.
# Let's define the exponential growth function
def exponential_growth(t, r):
    return np.exp(r * t)

# Fit the exponential growth model to the data. 
# We'll use a handy function from scipy called CURVE_FIT that allows us to fit any given function to our data. 
# We will fit the exponential growth function to the active cases data. HINT: Look up the documentation for curve_fit to see how to use it.

#parameters optizmized and parameter covariance
popt, pcov = curve_fit(exponential_growth, t_data, y_data)
r_fit = popt[0]
# Approximate R0 using this fit
R0_daily = np.exp(r_fit)

print(f"Growth Rate (r): {r_fit:.4f}")
print(f"Approximate Daily R0: {R0_daily:.4f}")

# Add the fit as a line on top of your scatterplot.

plt.figure(figsize=(10, 6))
plt.scatter(t_data, y_data, label='Reported Data', color='blue', alpha=0.5)
plt.plot(t_data, exponential_growth(t_data, r_fit), label='Exponential Fit', color='red', linewidth=2)
plt.xlabel('Day')
plt.ylabel('Active Reported Daily Cases')
plt.title('Mystery Virus Growth Analysis')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

# What viruses have a similar R0? Use the viruses.html file to find a virus or 2 with a similar R0 and give a 1-2 sentence background of the diseases.
# Seasonal Influenza 

# Seasonal Influenza is the virus that has the closest R0 to 1.12, tt's a common seasonal flu that has an R0 of 1.3. Known as a respiratory infection characterized by fever, cough, and muscle aches that circulates globally every year.

# How accurate do you think your R0 estimate is?
# Only moderately accurate since it assumes perfect exponential growth and has not taken account for real-world factors that might affect it. While it's useful for approximation, it's likely different from the true R0.




<div style="
    border-left: 6px solid #fbc02d;
    background-color: #fff8e1;
    padding: 10px 15px;
    border-radius: 4px;
">
<b style="color:#f57f17;">ANALYSIS AFTER DATA RELEASE #2</b> 

</div>



### 2c. Use Euler's method to solve the SEIR model.
This section should come from your python code after Data Release #2.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# LOAD DATA
data = pd.read_csv('Data/mystery_virus_daily_active_counts_RELEASE#2.csv', parse_dates=['date'], header=0, index_col=None)

t_data = data['day'].values
y_data = data['active reported daily cases'].values

# CONSTANTS
N = 17900
R0_daily = 1.1209  # Growth factor calculated from data
r = np.log(R0_daily) 
I0_start = y_data[0]

### 2c. Use Euler's method to solve the SEIR model.
def seir_euler(beta, sigma, gamma, S0, E0, I0, days, N, dt=0.1):
    steps = int(days / dt)
    S, E, I, R = [S0], [E0], [I0], [0.0]
    T = np.linspace(0, days, steps + 1)
    
    for i in range(steps):
        s_c, e_c, i_c, r_c = S[-1], E[-1], I[-1], R[-1]
        
        # Differential equations
        dS = -beta * s_c * i_c / N
        dE = (beta * s_c * i_c / N) - (sigma * e_c)
        dI = (sigma * e_c) - (gamma * i_c)
        dR = (gamma * i_c)
        
        # Update values
        S.append(s_c + dS * dt)
        E.append(e_c + dE * dt)
        I.append(i_c + dI * dt)
        R.append(r_c + dR * dt)
        
    # Interpolate back to daily integers for comparison
    I_daily = np.interp(np.arange(1, days + 1), T, I)
    return I_daily

### 2d. Fit the SEIR model to the data by changing beta, gamma, and sigma.
This section should come from your python code after Data Release #2.

In [ ]:
Day 10 → 3 cases
Day 30 → 38 cases
Growth rate formula:
r
=
ln
⁡
(
I
2
/
I
1
)
t
2
−
t
1
r= 
t 
2
​	
 −t 
1
​	
 
ln(I 
2
​	
 /I 
1
​	
 )
​	
 
r
=
ln
⁡
(
38
/
3
)
20
r= 
20
ln(38/3)
​	
 
r
=
ln
⁡
(
12.67
)
20
r= 
20
ln(12.67)
​	
 
r
≈
2.54
20
r≈ 
20
2.54
​	
 
r
≈
0.127
r≈0.127

In [ ]:
### 2d. Fit the SEIR model to the data by changing beta, gamma, and sigma.
def grid_search_seir(timepoints, N, S0, E0, I0, R0, data, 
                     beta_range, sigma_range, gamma_range):
    best_SSE = float("inf")
    best_params = None
    r_val = np.log(R0)
    
    for b in beta_range:
        for s in sigma_range:
            for g in gamma_range:
                # Calculate E0 based on the growth rate to stay consistent with early data
                E0_iter = ((r_val + g) / s) * I0
                S0_iter = N - I0 - E0_iter
                
                # Run Euler 
                I_pred = seir_euler(b, s, g, S0_iter, E0_iter, I0, int(timepoints[-1]), N)
                
                # Calculate SSE
                SSE = np.sum((I_pred - data)**2)
                
                if SSE < best_SSE:
                    best_SSE = SSE
                    best_params = (b, s, g, E0_iter)
                    
    return best_params, best_SSE

# RUN GRID SEARCH
beta_space = np.linspace(0.4, 0.9, 20)
sigma_space = np.linspace(1/18, 1/12, 10)
gamma_space = np.linspace(1/11, 1/7, 10)

best_p, min_sse = grid_search_seir(t_data, N, 0, 0, I0_start, R0_daily, y_data,
                                   beta_space, sigma_space, gamma_space)

b_opt, s_opt, g_opt, e0_opt = best_p
s0_opt = N - I0_start - e0_opt

### 2e. Plot the model-predicted infections over time compared to the data.
This section should come from your python code after Data Release #2.

In [ ]:
### 2e. Plot the model-predicted infections over time compared to the data.
# Extended simulation to 100 days to see the peak
t_proj_limit = 100
I_proj = seir_euler(b_opt, s_opt, g_opt, s0_opt, e0_opt, I0_start, t_proj_limit, N)
t_proj_days = np.arange(1, len(I_proj) + 1)

plt.figure(figsize=(10, 6))
plt.scatter(t_data, y_data, color='black', label='Reported Data', s=15, alpha=0.7)
plt.plot(t_proj_days, I_proj, color='red', label='Optimized SEIR Model', linewidth=2)
plt.xlabel('Day')
plt.ylabel('Active Infectious Cases')
plt.title('SEIR Model Fit and Epidemic Projection')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

### 2e. Predict the day and amount of active cases at the peak of the epidemic spread.
This section should come from your python code after Data Release #2.


In [ ]:
### 2e. Predict the day and amount of active cases at the peak of the epidemic spread.
peak_idx = np.argmax(I_proj)
peak_day = t_proj_days[peak_idx]
peak_cases = I_proj[peak_idx]

print(f"--- Results ---")
print(f"Minimum SSE: {min_sse:.2f}")
print(f"Predicted Peak Day: Day {peak_day}")
print(f"Predicted Peak Active Cases: {int(peak_cases)} cases")


<div style="
    border-left: 6px solid #fbc02d;
    background-color: #fff8e1;
    padding: 10px 15px;
    border-radius: 4px;
">
<b style="color:#f57f17;">ANALYSIS AFTER DATA RELEASE #3</b> 

</div>



### 2f. Plot the full dataset (Data Release #3) against your model.
This section should come from your python code after Data Release #3.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
# Load dataset
df = pd.read_csv('Data/mystery_virus_daily_active_counts_RELEASE#3.csv', parse_dates=['date'], header=0, index_col=None)

# Plotting
plt.figure(figsize=(10, 6))
plt.plot(df['day'], df['active reported daily cases'], color='blue', marker='o', linestyle='-', markersize=4, label='Active Reported Cases')

plt.xlabel('Day')
plt.ylabel('Active Reported Daily Cases')
plt.title('Mystery Virus Daily Active Counts (Release #3)')
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend()
plt.show()

### 2g. Intervention strategies for new outbreak at VT (70 days of infection)
This section should come from your python code after Data Release #3.



## Verify and validate your analysis: 

*(Describe how you checked to see that your analysis gave you an answer that you believe (verify). Describe how your determined if your analysis gave you an answer that is supported by other evidence (e.g., a published paper).*

## Conclusions and Ethical Implications: 
*(Think about the answer your analysis generated, draw conclusions related to your overarching question, and discuss the ethical implications of your conclusions.*

## Limitations and Future Work: 
*(Think about the answer your analysis generated, draw conclusions related to your overarching question, and discuss the ethical implications of your conclusions.*